In [76]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("Environment is ready")
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)


#Load Dataset
from pathlib import Path

data_path = Path("../data/raw/diabetic_data.csv")

df = pd.read_csv(data_path)

print("Dataset loaded successfully")
print("Rows and columns:", df.shape)


df.head()
df.columns.tolist()
df.info()

#Check the target variable
df["readmitted"].value_counts()

df["readmitted"].value_counts(normalize=True).mul(100).round(2)


#create binary target
df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)
df["readmitted_30"].value_counts()

#Replace "?" with missing values
import numpy as np
df = df.replace("?", np.nan)
print("Question marks replaced with missing values")

#Check Missing Values

missing_counts = df.isnull().sum().sort_values(ascending=False)
missing_counts.head(15)

missing_percent = (
    df.isnull()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .round(2)
)

missing_percent.head(15)
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": df.isnull().mean().mul(100).round(2)
})

missing_summary = missing_summary.sort_values(
    by="missing_percent",
    ascending=False
)

missing_summary.head(15)

#Check repeated patients
print("Total encounters:", len(df))
print("Unique patients:", df["patient_nbr"].nunique())

#Count patients have multiple encounters
patient_counts = df["patient_nbr"].value_counts()

print("Patients with more than one encounter:", (patient_counts > 1).sum())
print("Maximum encounters for one patient:", patient_counts.max())

#Check duplicates
print("Exact duplicate rows:", df.duplicated().sum())
print("Unique encounter IDs:", df["encounter_id"].nunique())
print("Total rows:", len(df))

#Original Target distribution
print(df["readmitted"].value_counts())

print(
    df["readmitted"]
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
)

#verify Binary target mapping
pd.crosstab(
    df["readmitted"],
    df["readmitted_30"],
    margins=True
)

#Unique values in each column
unique_values = df.nunique().sort_values()

unique_values

#Identify constant columns
constant_columns = [
    col for col in df.columns
    if df[col].nunique(dropna=False) == 1
]

print("Constant columns:", constant_columns)
print("Number of constant columns:", len(constant_columns))

#check rare categories
for col in ["gender", "race", "age"]:
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False))

#check gender categories
df["gender"].value_counts(dropna=False)

#Check race categories
df["race"].value_counts(dropna=False)

#Check age distribution completely
df["age"].value_counts().sort_index()

#Check missingness in the diagnosis columns
for col in ["diag_1", "diag_2", "diag_3"]:
    print(f"\n{col}")
    print("Missing count:", df[col].isna().sum())
    print("Unique values:", df[col].nunique())

#three diagnosis columns
diagnosis_summary = pd.DataFrame({
    "missing_count": df[["diag_1", "diag_2", "diag_3"]].isna().sum(),
    "missing_percent": (
        df[["diag_1", "diag_2", "diag_3"]]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    ),
    "unique_values": df[["diag_1", "diag_2", "diag_3"]].nunique()
})

diagnosis_summary

#Inspect numerical feature ranges
numerical_columns = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

df[numerical_columns].describe().T

#Check skewness and extreme percentiles
utilization_columns = [
    "number_outpatient",
    "number_emergency",
    "number_inpatient"
]

summary = pd.DataFrame({
    "zero_percent": (
        df[utilization_columns]
        .eq(0)
        .mean()
        .mul(100)
        .round(2)
    ),
    "95th_percentile": df[utilization_columns].quantile(0.95),
    "99th_percentile": df[utilization_columns].quantile(0.99),
    "maximum": df[utilization_columns].max()
})

summary

#Check missing and rare medical-specialty values
print("Missing medical specialty:", df["medical_specialty"].isna().sum())
print("Unique specialties:", df["medical_specialty"].nunique())

df["medical_specialty"].value_counts(dropna=False).head(15)

#Count how many specialties are rare
specialty_counts = df["medical_specialty"].value_counts()

print("Specialties with fewer than 100 encounters:",
      (specialty_counts < 100).sum())

print("Specialties with fewer than 500 encounters:",
      (specialty_counts < 500).sum())

#Inspect payer_code
print("Missing payer code:", df["payer_code"].isna().sum())
print("Unique payer codes:", df["payer_code"].nunique())

df["payer_code"].value_counts(dropna=False)

#Inspect Weight
print("Missing weight:", df["weight"].isna().sum())
print("Missing percent:", round(df["weight"].isna().mean() * 100, 2))
print("Unique weight categories:", df["weight"].nunique())

df["weight"].value_counts(dropna=False)

#Inspect A1Cresult and max_glu_serum
for col in ["A1Cresult", "max_glu_serum"]:
    print(f"\nColumn: {col}")
    print("Missing count:", df[col].isna().sum())
    print("Missing percent:", round(df[col].isna().mean() * 100, 2))
    print(df[col].value_counts(dropna=False))
df["A1Cresult"].value_counts(dropna=False)

#Inspect medication-feature variation
medication_columns = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin"
]

medication_summary = pd.DataFrame({
    col: df[col].value_counts(dropna=False)
    for col in medication_columns
}).T.fillna(0).astype(int)

medication_summary

#Check change and diabetesMed
for col in ["change", "diabetesMed"]:
    print(f"\nColumn: {col}")
    print(df[col].value_counts(dropna=False))

#Check change
df["change"].value_counts(dropna=False)

#Check discharge disposition categories
df["discharge_disposition_id"].value_counts().sort_index()

#Load the discharge-ID descriptions
import pandas as pd
mapping_df = pd.read_csv("../data/raw/IDs_mapping.csv")
mapping_df.head(20)

#discharge mapping
start_index = mapping_df[
    mapping_df["admission_type_id"] == "discharge_disposition_id"
].index[0]

end_index = mapping_df[
    mapping_df["admission_type_id"] == "admission_source_id"
].index[0]

discharge_mapping = mapping_df.loc[
    start_index + 1:end_index - 1,
    ["admission_type_id", "description"]
].copy()

discharge_mapping.columns = [
    "discharge_disposition_id",
    "description"
]

discharge_mapping

#Exclude dischargeids for expired patients
excluded_discharge_ids = [11, 13, 14, 19, 20, 21]

df["discharge_disposition_id"].isin(excluded_discharge_ids).value_counts()

# Modeling Dataset by Excluding Ineligible Discharge Outcomes
excluded_discharge_ids = [11, 13, 14, 19, 20, 21]

df_model = df[
    ~df["discharge_disposition_id"].isin(excluded_discharge_ids)
].copy()

print("Original rows:", len(df))
print("Rows after exclusion:", len(df_model))
print("Rows removed:", len(df) - len(df_model))

#Recheck the Target Distribution After Exclusions
target_after_exclusion = pd.DataFrame({
    "count": df_model["readmitted_30"].value_counts(),
    "percentage": (
        df_model["readmitted_30"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )
})

target_after_exclusion

#Recheck Unique Patients After Discharge Exclusions
print("Total modeling encounters:", len(df_model))
print("Unique patients in modeling data:", df_model["patient_nbr"].nunique())

patient_counts_model = df_model["patient_nbr"].value_counts()

print(
    "Patients with multiple encounters:",
    (patient_counts_model > 1).sum()
)

print(
    "Maximum encounters for one patient:",
    patient_counts_model.max()
)

#Inspect Admission Type Categories
df_model["admission_type_id"].value_counts().sort_index()

#Interpret Admission Type Distribution
df_model["admission_source_id"].value_counts().sort_index()

#Extract the Admission Source Mapping
start_index = mapping_df[
    mapping_df["admission_type_id"] == "admission_source_id"
].index[0]

admission_source_mapping = mapping_df.loc[
    start_index + 1:,
    ["admission_type_id", "description"]
].copy()

admission_source_mapping.columns = [
    "admission_source_id",
    "description"
]

admission_source_mapping = admission_source_mapping.dropna(
    subset=["admission_source_id"]
)

admission_source_mapping

#Convert Admission Source IDs into Readable Categories
# Convert mapping IDs to numeric values
admission_source_mapping["admission_source_id"] = pd.to_numeric(
    admission_source_mapping["admission_source_id"],
    errors="coerce"
)

# Create an ID-to-description dictionary
admission_source_dict = dict(
    zip(
        admission_source_mapping["admission_source_id"],
        admission_source_mapping["description"]
    )
)

# Add readable admission-source descriptions
df_model["admission_source_desc"] = (
    df_model["admission_source_id"]
    .map(admission_source_dict)
    .fillna("Unknown")
)

df_model["admission_source_desc"].value_counts(dropna=False)

#Check Rare Admission Source Categories
admission_source_counts = df_model[
    "admission_source_desc"
].value_counts()

print(
    "Categories with fewer than 100 encounters:",
    (admission_source_counts < 100).sum()
)

print(
    "Categories with fewer than 500 encounters:",
    (admission_source_counts < 500).sum()
)

#Create Grouped Admission Source Categories
# Remove extra spaces from the descriptions
df_model["admission_source_desc"] = (
    df_model["admission_source_desc"]
    .astype(str)
    .str.strip()
)

def group_admission_source(source):
    if source == "Emergency Room":
        return "Emergency Room"

    if source in [
        "Physician Referral",
        "Clinic Referral",
        "HMO Referral"
    ]:
        return "Referral"

    if source in [
        "Transfer from a hospital",
        "Transfer from a Skilled Nursing Facility (SNF)",
        "Transfer from another health care facility",
        "Transfer from critial access hospital",
        "Transfer from hospital inpt/same fac reslt in a sep claim",
        "Transfer from Ambulatory Surgery Center"
    ]:
        return "Transfer"

    if source in [
        "Unknown",
        "Not Mapped",
        "Not Available",
        "Unknown/Invalid",
        "nan"
    ]:
        return "Unknown"

    return "Other"


df_model["admission_source_group"] = (
    df_model["admission_source_desc"]
    .apply(group_admission_source)
)

df_model["admission_source_group"].value_counts()

#Create Readable Admission Type Groups
admission_type_map = {
    1: "Emergency",
    2: "Urgent",
    3: "Elective",
    4: "Other",
    5: "Unknown",
    6: "Unknown",
    7: "Other",
    8: "Unknown"
}

df_model["admission_type_group"] = (
    df_model["admission_type_id"]
    .map(admission_type_map)
    .fillna("Unknown")
)

df_model["admission_type_group"].value_counts()

#Create a Preliminary Column Decision Table
column_decisions = pd.DataFrame({
    "column": [
        "encounter_id",
        "patient_nbr",
        "weight",
        "payer_code",
        "examide",
        "citoglipton",
        "A1Cresult",
        "max_glu_serum",
        "medical_specialty",
        "diag_1",
        "diag_2",
        "diag_3",
        "admission_type_id",
        "admission_source_id"
    ],
    "decision": [
        "Drop from model",
        "Use only for patient-level split",
        "Drop",
        "Likely drop",
        "Drop",
        "Drop",
        "Keep and label missing as Not measured",
        "Keep and label missing as Not measured",
        "Keep, fill Unknown, group rare categories",
        "Keep but group ICD codes",
        "Keep but group ICD codes",
        "Keep but group ICD codes",
        "Replace with admission_type_group",
        "Replace with admission_source_group"
    ],
    "reason": [
        "Unique identifier",
        "Prevents patient leakage",
        "96.86% missing",
        "39.56% missing and limited clinical usefulness",
        "Constant column",
        "Constant column",
        "Test result may be informative",
        "Test result may be informative",
        "Potentially useful but high missingness and cardinality",
        "High cardinality diagnosis codes",
        "High cardinality diagnosis codes",
        "High cardinality diagnosis codes",
        "Readable grouped categories created",
        "Readable grouped categories created"
    ]
})

column_decisions


#Save the Column Decision Table
from pathlib import Path

decision_path = Path("../outputs/metrics/column_decisions.csv")

column_decisions.to_csv(decision_path, index=False)

print("Column decision table saved to:", decision_path)

#Save the Current Modeling Dataset
from pathlib import Path

modeling_data_path = Path(
    "../data/processed/diabetic_modeling_data.csv"
)

df_model.to_csv(modeling_data_path, index=False)

print("Modeling dataset saved to:", modeling_data_path)
print("Saved shape:", df_model.shape)

#Create Diagnosis Groups
def group_diagnosis(code):
    if pd.isna(code):
        return "Unknown"

    code = str(code).strip()

    if code.startswith(("V", "E")):
        return "Other"

    try:
        code_num = float(code)
    except ValueError:
        return "Other"

    if 390 <= code_num <= 459 or code_num == 785:
        return "Circulatory"

    if 460 <= code_num <= 519 or code_num == 786:
        return "Respiratory"

    if 520 <= code_num <= 579 or code_num == 787:
        return "Digestive"

    if 250 <= code_num < 251:
        return "Diabetes"

    if 800 <= code_num <= 999:
        return "Injury"

    if 710 <= code_num <= 739:
        return "Musculoskeletal"

    if 580 <= code_num <= 629 or code_num == 788:
        return "Genitourinary"

    if 140 <= code_num <= 239:
        return "Neoplasms"

    return "Other"


for col in ["diag_1", "diag_2", "diag_3"]:
    df_model[f"{col}_group"] = df_model[col].apply(group_diagnosis)

df_model[
    ["diag_1_group", "diag_2_group", "diag_3_group"]
].head()

#Validate the Diagnosis Group Distributions
for col in ["diag_1_group", "diag_2_group", "diag_3_group"]:
    print(f"\n{col}")
    print(df_model[col].value_counts(dropna=False))

#Display Diagnosis Group Counts in One Compact Table
diagnosis_group_summary = pd.DataFrame({
    "diag_1_count": df_model["diag_1_group"].value_counts(),
    "diag_2_count": df_model["diag_2_group"].value_counts(),
    "diag_3_count": df_model["diag_3_group"].value_counts()
}).fillna(0).astype(int)

diagnosis_group_summary

#Clean and Group Medical Specialty
df_model["medical_specialty_clean"] = (
    df_model["medical_specialty"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

specialty_counts = df_model["medical_specialty_clean"].value_counts()

common_specialties = specialty_counts[
    specialty_counts >= 500
].index

df_model["medical_specialty_group"] = (
    df_model["medical_specialty_clean"]
    .where(
        df_model["medical_specialty_clean"].isin(common_specialties),
        "Other"
    )
)

df_model["medical_specialty_group"].value_counts()

#Fill Remaining Missing Categorical Values
columns_to_fill = [
    "race",
    "A1Cresult",
    "max_glu_serum",
    "diag_1",
    "diag_2",
    "diag_3"
]

for col in columns_to_fill:
    df_model[col] = df_model[col].fillna("Unknown")

print(
    df_model[columns_to_fill]
    .isnull()
    .sum()
)



df_model[
    [
        "race",
        "A1Cresult",
        "max_glu_serum",
        "diag_1",
        "diag_2",
        "diag_3"
    ]
].isnull().sum()

#Remove Confirmed Unusable Columns
columns_to_drop = [
    "weight",
    "examide",
    "citoglipton"
]

df_model = df_model.drop(
    columns=columns_to_drop,
    errors="ignore"
)

print("Dropped columns:", columns_to_drop)
print("Updated shape:", df_model.shape)

#Check Remaining Missing Values in the Modeling Dataset
remaining_missing = (
    df_model.isnull()
    .sum()
    .sort_values(ascending=False)
)

remaining_missing[remaining_missing > 0]

#Remove Original Columns Replaced by Cleaned Versions
columns_to_remove = [
    "medical_specialty",
    "medical_specialty_clean",
    "payer_code"
]

df_model = df_model.drop(
    columns=columns_to_remove,
    errors="ignore"
)

print("Removed columns:", columns_to_remove)
print("Updated shape:", df_model.shape)

print(
    "Remaining missing values:",
    df_model.isnull().sum().sum()
)

#Review the Final Dataset Columns
for index, column in enumerate(df_model.columns, start=1):
    print(f"{index}. {column}")

#Identify Original and Engineered Duplicate Columns
review_columns = [
    col for col in df_model.columns
    if any(keyword in col for keyword in [
        "admission",
        "diag",
        "specialty",
        "readmitted",
        "encounter",
        "patient"
    ])
]

for col in review_columns:
    print(col)

#Remove Redundant Original and Helper Columns
redundant_columns = [
    "admission_type_id",
    "admission_source_id",
    "admission_source_desc",
    "diag_1",
    "diag_2",
    "diag_3",
    "readmitted"
]

df_model = df_model.drop(
    columns=redundant_columns,
    errors="ignore"
)

print("Removed redundant columns:", redundant_columns)
print("Final audit dataset shape:", df_model.shape)

#Final Validation and Save the Completed Audit Dataset
print("Final shape:", df_model.shape)
print("Total missing values:", df_model.isnull().sum().sum())
print("Duplicate rows:", df_model.duplicated().sum())

print(
    "Target distribution:\n",
    df_model["readmitted_30"].value_counts()
)

print(
    "Target percentage:\n",
    df_model["readmitted_30"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

#Save the Final Audited Dataset
from pathlib import Path

final_audit_path = Path(
    "../data/processed/diabetic_modeling_data_final.csv"
)

df_model.to_csv(final_audit_path, index=False)

print("Final audited dataset saved to:", final_audit_path)
print("Final saved shape:", df_model.shape)
print("Remaining missing values:", df_model.isnull().sum().sum())








Environment is ready
Pandas version: 3.0.3
Scikit-learn version: 1.9.0
Dataset loaded successfully
Rows and columns: (101766, 50)
<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      101766 non-null  str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    101766 non-null  str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                101766 non-null  str  
 11  medical_specialty         101766 non-null  str  
 12  num_lab_p